# momentum ceiling experiment

extends the v3 ceiling finding to surface u and v not just T. if v3 ML surface u/v match v3 BULK, the ML isn't the bottleneck for momentum either. strengthens the 'sim architecture is the binding constraint' story across all variables.

In [ ]:
using JLD2
using Plots

gr()
default(size=(900,500), linewidth=2)

const ROOT = normpath(joinpath(@__DIR__, ".."))
const SITES = ["lat30lon-50","lat-25lon-10","lat-45lon80","lat0lon-140","lat30lon-150","lat40lon-25"]

load_v3(site) = JLD2.load(joinpath(ROOT, "output/era5/ml_forced_30day_v3_qt_$(site).jld2"))
load_v3b(site) = JLD2.load(joinpath(ROOT, "output/era5/ml_forced_30day_v3bulk_qt_$(site).jld2"))

## surface u — v3 ML vs v3 BULK

In [ ]:
function panel_u(site)
    v3 = load_v3(site); v3b = load_v3b(site)
    t = v3["saved_times"] ./ 86400
    plt = plot(title=site, xlabel="Day", ylabel="surface u (m/s)", titlefontsize=10)
    plot!(plt, t, [pr[end] for pr in v3["u_profiles"]], label="v3 ML", color=:red, lw=2)
    plot!(plt, t, [pr[end] for pr in v3b["u_profiles"]], label="v3 BULK", color=:purple, lw=2, linestyle=:dash)
    plt
end
plot([panel_u(s) for s in SITES]..., layout=(3,2), size=(1200,900))

## surface v — v3 ML vs v3 BULK

In [ ]:
function panel_v(site)
    v3 = load_v3(site); v3b = load_v3b(site)
    t = v3["saved_times"] ./ 86400
    plt = plot(title=site, xlabel="Day", ylabel="surface v (m/s)", titlefontsize=10)
    plot!(plt, t, [pr[end] for pr in v3["v_profiles"]], label="v3 ML", color=:red, lw=2)
    plot!(plt, t, [pr[end] for pr in v3b["v_profiles"]], label="v3 BULK", color=:purple, lw=2, linestyle=:dash)
    plt
end
plot([panel_v(s) for s in SITES]..., layout=(3,2), size=(1200,900))

## momentum ceiling gap u and v

In [ ]:
plt_u = plot(xlabel="Day", ylabel="u: v3 ML − v3 BULK (m/s)", title="u ceiling gap", legend=:outertopright)
plt_v = plot(xlabel="Day", ylabel="v: v3 ML − v3 BULK (m/s)", title="v ceiling gap", legend=:outertopright)
for s in SITES
    v3 = load_v3(s); v3b = load_v3b(s)
    N = min(length(v3["u_profiles"]), length(v3b["u_profiles"]))
    t = v3["saved_times"][1:N] ./ 86400
    du = [v3["u_profiles"][i][end] - v3b["u_profiles"][i][end] for i in 1:N]
    dv = [v3["v_profiles"][i][end] - v3b["v_profiles"][i][end] for i in 1:N]
    plot!(plt_u, t, du, label=s, lw=1.5)
    plot!(plt_v, t, dv, label=s, lw=1.5)
end
hline!(plt_u, [0], color=:black, linestyle=:dot, label="zero")
hline!(plt_v, [0], color=:black, linestyle=:dot, label="zero")
plot(plt_u, plt_v, layout=(2,1), size=(1000,800))

## day 30 table all 3 variables

In [ ]:
using Printf
println(rpad("site",14), "| ΔT (ML-BULK) | Δu           | Δv")
println("-"^60)
for s in SITES
    v3 = load_v3(s); v3b = load_v3b(s)
    dT = v3["T_profiles"][end][end] - v3b["T_profiles"][end][end]
    du = v3["u_profiles"][end][end] - v3b["u_profiles"][end][end]
    dv = v3["v_profiles"][end][end] - v3b["v_profiles"][end][end]
    @printf("%-14s| %+9.4f     | %+9.5f    | %+9.5f\n", s, dT, du, dv)
end